# EDA: Calidad de Red

## Caso Práctico - Empresa de Telecomunicaciones
## MBI — Prácticas Aplicadas 2026

---

En este notebook analizamos la tabla de calidad de señal por zona y mes.

A diferencia de las otras fuentes, esta tabla **no está al nivel del cliente** sino al nivel de zona-mes. Esto significa que no podemos saber exactamente qué calidad de red experimenta cada cliente individualmente, pero sí podemos saber la calidad de la zona donde vive.

Según el profesor, la calidad de red es uno de los drivers principales del churn:
- Mala calidad → menos consumo
- Mala calidad → más impagos
- Mala calidad → más llamadas a soporte
- Todo eso → más churn

**Fuente:** `calidad_senal_zona_mensual.csv`

**Variables:**
- `cobertura_4g_pct` / `cobertura_5g_pct` → % de cobertura por tecnología
- `latencia_ms` → tiempo de respuesta de la red (ms)
- `velocidad_media_mbps` → velocidad media de descarga
- `tasa_cortes_pct` → % de llamadas/conexiones con corte
- `indice_calidad_global` → índice sintético de calidad (más alto = mejor)
- `incidencia_masiva` → si hubo una incidencia masiva ese mes en esa zona


## Objetivos

1. Explorar la estructura y calidad del dataset
2. Detectar problemas de calidad de datos (errores tipográficos, outliers, nulos)
3. Contrastar hipótesis sobre la calidad de red:
   - **H1**: ¿Las zonas rurales tienen peor calidad de red?
   - **H2**: ¿Hay regiones con peor calidad estructural?
   - **H3**: ¿La cobertura 5G ha mejorado con el tiempo?
   - **H4**: ¿Las incidencias masivas se asocian a picos de latencia y cortes?
   - **H5**: ¿La calidad de red de la zona está relacionada con el churn del cliente?
4. Clasificación simple para validar H5 cuantitativamente


## Nota metodológica

Para H5 necesitamos cruzar esta tabla con los clientes y el churn. El cruce se hace por `zona_id`: cada cliente tiene asignada una zona en `clientes.csv`, y usamos la calidad media de esa zona como proxy de la experiencia del cliente.

Es una aproximación, porque dos clientes en la misma zona pueden tener experiencias distintas (distintas calles, distintos dispositivos...), pero es la mejor que podemos hacer con los datos disponibles.


## 1. Librerías


In [ ]:
import warnings
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
sns.set_theme(style='whitegrid', context='notebook')

# Añadimos la raíz del proyecto al path para poder importar src/
ROOT = Path('..').resolve()
sys.path.append(str(ROOT))

from src.utils.helpers import (
    resumen_df, test_mannwhitney, cramers_v,
    boxplot_churn, kde_churn, barras_churn_cat, perfil_churner
)

PAL = {'No Churn': '#4C9BE8', 'Churn': '#E85C4C'}
print('Librerías cargadas')

## 2. Carga de datos


In [ ]:
# Cargamos usando los módulos del proyecto
from src.load import load_calidad, load_clientes, load_churn

calidad  = load_calidad()
clientes = load_clientes()
churn    = load_churn()

print('\nDatasets cargados correctamente')

---
## 3. Exploración inicial y calidad de datos

Antes de cualquier hipótesis, miramos qué tenemos y si hay problemas.


In [ ]:
resumen_df(calidad, 'calidad_senal_zona_mensual.csv')
display(calidad.head())

In [ ]:
display(calidad.describe())

### 3.1 Problemas de calidad detectados

Mirando el describe ya se ven cosas raras que hay que investigar antes de seguir.


In [ ]:
# 1. Tipos de zona: ¿hay errores tipográficos?
print('Valores únicos en tipo_zona:')
print(calidad['tipo_zona'].value_counts())
print()

# 2. Velocidad negativa — imposible físicamente
print(f'Registros con velocidad_media_mbps < 0: {(calidad["velocidad_media_mbps"] < 0).sum()}')
print(calidad[calidad['velocidad_media_mbps'] < 0][['fecha','zona_id','tipo_zona','velocidad_media_mbps']])
print()

# 3. Latencia extrema (> 200ms es ya muy alta en una red móvil)
print(f'Registros con latencia > 200ms: {(calidad["latencia_ms"] > 200).sum()}')
print(calidad[calidad['latencia_ms'] > 200][['fecha','zona_id','tipo_zona','latencia_ms']])
print()

# 4. Nulos
print('Nulos por columna:')
print(calidad.isnull().sum())

### Hallazgos de calidad

**Errores tipográficos en `tipo_zona`**: aparecen valores como `suburbanx`, `urbana??` y `rural-1` que son errores de introducción de datos. En lugar de gestionarlos aquí, la corrección está centralizada en `src/clean.py` dentro de `clean_calidad()`, que es donde corresponde hacer la limpieza. De esta forma el notebook recibe los datos ya limpios y no mezcla análisis con limpieza.

**Velocidad negativa**: hay registros con `velocidad_media_mbps < 0`, físicamente imposible. También se trata en `clean_calidad()` → se convierte a NaN.

**Latencia extrema**: un registro con más de 500ms. Puede ser una incidencia puntual real o un error de medición. También tratado en `clean_calidad()` → NaN.

⚠️ En el notebook trabajamos directamente con el CSV raw para poder mostrar los problemas, pero aplicamos la misma corrección localmente para el análisis.


In [ ]:
# Aplicamos aquí la misma limpieza que está centralizada en clean.py
# En el pipeline real (main.py) esto ya viene hecho automáticamente
calidad_clean = calidad.copy()

# Normalizar tipo_zona — corregir errores tipográficos
mapa_zona = {
    'suburbanx': 'suburbana',
    'urbana??':  'urbana_premium',
    'rural-1':   'rural',
}
calidad_clean['tipo_zona'] = calidad_clean['tipo_zona'].replace(mapa_zona)
print('Tipos de zona tras limpieza:', sorted(calidad_clean['tipo_zona'].unique()))

# Velocidad negativa → NaN
mask_vel = calidad_clean['velocidad_media_mbps'] < 0
calidad_clean.loc[mask_vel, 'velocidad_media_mbps'] = None
print(f'Velocidades negativas corregidas: {mask_vel.sum()}')

# Latencia > 300ms → NaN
mask_lat = calidad_clean['latencia_ms'] > 300
calidad_clean.loc[mask_lat, 'latencia_ms'] = None
print(f'Latencias extremas corregidas: {mask_lat.sum()}')

### 3.2 Distribución de las variables de calidad


In [ ]:
vars_num = ['cobertura_4g_pct', 'cobertura_5g_pct', 'latencia_ms',
            'velocidad_media_mbps', 'tasa_cortes_pct', 'indice_calidad_global']

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()

for i, col in enumerate(vars_num):
    datos = calidad_clean[col].dropna()
    axes[i].hist(datos, bins=30, color='steelblue', edgecolor='white', alpha=0.8)
    axes[i].axvline(datos.mean(), color='red', linestyle='--',
                    label=f'Media: {datos.mean():.1f}')
    axes[i].axvline(datos.median(), color='orange', linestyle='--',
                    label=f'Mediana: {datos.median():.1f}')
    axes[i].set_title(f'Distribución de {col}', fontweight='bold')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Frecuencia')
    axes[i].legend(fontsize=8)

plt.suptitle('Distribución de variables de calidad de red', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 4. Análisis de Hipótesis


### H1 — ¿Las zonas rurales tienen peor calidad de red?

Es lógico esperar que las zonas rurales tengan menos infraestructura de red. Si esto se confirma, el tipo de zona será una variable importante para explicar el churn en esas áreas.


In [ ]:
orden_zonas = ['rural', 'suburbana', 'urbana_premium']
vars_calidad = ['indice_calidad_global', 'cobertura_4g_pct',
                'cobertura_5g_pct', 'latencia_ms',
                'velocidad_media_mbps', 'tasa_cortes_pct']

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

for i, var in enumerate(vars_calidad):
    data_plot = calidad_clean[calidad_clean['tipo_zona'].isin(orden_zonas)]
    sns.boxplot(data=data_plot, x='tipo_zona', y=var,
                order=orden_zonas,
                palette=sns.color_palette('Blues', 3),
                ax=axes[i])
    axes[i].set_title(f'{var} por tipo de zona', fontweight='bold')
    axes[i].set_xlabel('Tipo de zona')
    axes[i].set_ylabel(var)

plt.suptitle('H1 — Calidad de red por tipo de zona', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Tabla resumen
resumen_zonas = (calidad_clean[calidad_clean['tipo_zona'].isin(orden_zonas)]
                 .groupby('tipo_zona')[vars_calidad]
                 .mean()
                 .reindex(orden_zonas)
                 .round(2))
display(resumen_zonas)

### Resultado H1

La hipótesis se confirma con claridad en todas las métricas:

Las zonas **rurales tienen peor calidad en todos los indicadores**: el índice de calidad global es de ~22 en rural frente a ~50 en urbana_premium, la cobertura 5G apenas llega al 13% en rural mientras que en urbana_premium supera el 40%, la velocidad media es de ~38 Mbps en rural frente a ~83 Mbps en urbana_premium, y la tasa de cortes es el doble (~3% vs ~1%).

La única variable donde las diferencias son más pequeñas es la latencia, donde rural tiene ~55ms y urbana_premium ~35ms, pero la diferencia sigue siendo notable.

Este resultado es importante de cara al modelo: **el tipo de zona y la calidad de red son variables complementarias**. Un cliente rural parte de una experiencia de red estructuralmente peor, lo que puede explicar parte de su mayor propensión al churn que ya vimos en el EDA de clientes.


### H2 — ¿Hay regiones con peor calidad estructural?

Más allá del tipo de zona, puede haber diferencias geográficas entre regiones: Norte, Sur, Este, Oeste, Centro. Si alguna región tiene peor infraestructura de forma sistemática, eso explicaría por qué tiene más churn.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Índice de calidad global por región
calidad_region = (calidad_clean.groupby('region')['indice_calidad_global']
                  .mean()
                  .sort_values())

bars = axes[0].barh(calidad_region.index, calidad_region.values,
                    color=sns.color_palette('RdYlGn', len(calidad_region)))
axes[0].set_title('Índice de calidad global medio por región', fontweight='bold')
axes[0].set_xlabel('Índice de calidad global (media)')
for bar, val in zip(bars, calidad_region.values):
    axes[0].text(val + 0.2, bar.get_y() + bar.get_height()/2,
                 f'{val:.1f}', va='center', fontsize=9)

# Heatmap: calidad media por región x tipo_zona
pivot = (calidad_clean[calidad_clean['tipo_zona'].isin(orden_zonas)]
         .groupby(['region', 'tipo_zona'])['indice_calidad_global']
         .mean()
         .unstack())
sns.heatmap(pivot, annot=True, fmt='.1f', cmap='RdYlGn',
            ax=axes[1], linewidths=0.5)
axes[1].set_title('Calidad global media — Región x Tipo de zona', fontweight='bold')
axes[1].set_xlabel('Tipo de zona')
axes[1].set_ylabel('Región')

plt.tight_layout()
plt.show()

### Resultado H2

A nivel de región, hay diferencias pero son **mucho más moderadas que las del tipo de zona**: Este y Oeste tienen el mejor índice (~40), mientras que Centro es la peor (28.5) y Norte tiene 31.1.

El heatmap revela algo muy relevante: **las diferencias entre regiones desaparecen casi por completo cuando controlamos por tipo de zona**. Rural siempre da ~22-23 independientemente de la región, y urbana_premium siempre da ~50-51. Esto significa que la región no añade información adicional una vez sabemos el tipo de zona.

La conclusión práctica es que **`tipo_zona` es la variable geográfica más relevante** para el modelo, no la región. Tener las dos puede generar redundancia.

La excepción es Centro, que no tiene zonas urbana_premium en el heatmap (celda vacía), lo que puede indicar que es una región sin zonas de alta densidad urbana.


### H3 — ¿La cobertura 5G ha mejorado con el tiempo?

El dataset cubre varios años. El 5G es una tecnología que se ha ido desplegando progresivamente. Si vemos una tendencia creciente en `cobertura_5g_pct`, confirmaríamos que la empresa ha estado invirtiendo en mejorar su red. Esto también puede afectar al churn: los clientes en zonas donde llegó el 5G tarde pueden haber abandonado antes.


In [ ]:
evol = calidad_clean.groupby('fecha').agg(
    cobertura_5g      = ('cobertura_5g_pct',       'mean'),
    cobertura_4g      = ('cobertura_4g_pct',       'mean'),
    velocidad         = ('velocidad_media_mbps',   'mean'),
    latencia          = ('latencia_ms',            'mean'),
    calidad_global    = ('indice_calidad_global',  'mean'),
    tasa_cortes       = ('tasa_cortes_pct',        'mean'),
).reset_index()

fig, axes = plt.subplots(3, 2, figsize=(16, 12), sharex=True)
axes = axes.flatten()

metricas = [
    ('cobertura_5g',   'Cobertura 5G (%)',          '#9b59b6'),
    ('cobertura_4g',   'Cobertura 4G (%)',          '#3498db'),
    ('velocidad',      'Velocidad media (Mbps)',     '#27ae60'),
    ('latencia',       'Latencia media (ms)',        '#e74c3c'),
    ('calidad_global', 'Índice calidad global',     '#f39c12'),
    ('tasa_cortes',    'Tasa de cortes (%)',         '#e67e22'),
]

for i, (col, titulo, color) in enumerate(metricas):
    axes[i].plot(evol['fecha'], evol[col], color=color, linewidth=2)
    axes[i].fill_between(evol['fecha'], evol[col], alpha=0.15, color=color)
    axes[i].set_title(titulo, fontweight='bold')
    axes[i].set_ylabel(titulo)
    axes[i].tick_params(axis='x', rotation=45)

plt.suptitle('H3 — Evolución temporal de las métricas de calidad de red',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### Resultado H3

La hipótesis se confirma: la **cobertura 5G ha crecido de forma constante** desde el 10% en enero de 2023 hasta el 43% a finales de 2025. Es la tendencia más clara de todo el análisis.

Acompañando al despliegue del 5G, también se observa una **mejora progresiva del índice de calidad global** (de ~28 a ~43) y un **aumento de la velocidad media** (de ~50 a ~65 Mbps). La latencia muestra una tendencia ligeramente decreciente, aunque con dos picos puntuales alrededor de mayo 2024 que probablemente coincidan con alguna incidencia.

La cobertura 4G se mantiene prácticamente plana (~88%), lo que tiene sentido: es una tecnología ya madura y bien desplegada.

Un aspecto importante para el modelo: como la calidad de red ha mejorado con el tiempo, **un cliente que churneó en 2023 vivió una experiencia de red peor que uno que churneó en 2025**. Esto es un argumento más para usar lags temporales en lugar de medias históricas.


### H4 — ¿Las incidencias masivas se asocian a picos de latencia y cortes?

Las incidencias masivas son eventos puntuales donde la red falla de forma generalizada en una zona. Es de esperar que en esos meses la latencia suba y la tasa de cortes también. Si esto se confirma, la variable `incidencia_masiva` es una señal importante para el modelo.


In [ ]:
calidad_clean['incidencia_label'] = calidad_clean['incidencia_masiva'].map(
    {0: 'Sin incidencia', 1: 'Con incidencia'}
)

vars_impacto = ['latencia_ms', 'tasa_cortes_pct',
                'velocidad_media_mbps', 'indice_calidad_global']

fig, axes = plt.subplots(1, 4, figsize=(18, 5))

for i, var in enumerate(vars_impacto):
    data_plot = calidad_clean.dropna(subset=[var, 'incidencia_label'])
    sns.boxplot(data=data_plot, x='incidencia_label', y=var,
                order=['Sin incidencia', 'Con incidencia'],
                palette={'Sin incidencia': '#4C9BE8', 'Con incidencia': '#E85C4C'},
                ax=axes[i])
    axes[i].set_title(f'{var}\nvs incidencia masiva', fontweight='bold')
    axes[i].set_xlabel('')

plt.suptitle('H4 — Impacto de las incidencias masivas en la calidad de red',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Tabla comparativa
print('\nMedia de métricas según incidencia masiva:')
display(calidad_clean.groupby('incidencia_masiva')[vars_impacto].mean().round(2))

### Resultado H4

Los resultados son llamativos a pesar de que solo hay 5 incidencias masivas en todo el dataset:

- La **tasa de cortes se dispara** en meses con incidencia (mediana ~7.5% vs ~2% sin incidencia)
- La **velocidad cae drásticamente** (mediana ~33 Mbps vs ~57 Mbps sin incidencia)
- El **índice de calidad global baja** considerablemente (mediana ~28 vs ~35)
- La latencia sube algo pero con más variabilidad

La dirección de todos los efectos es la esperada, lo que da credibilidad a los datos aunque la muestra sea pequeña.

Hay que mencionar que con solo 5 incidencias el test estadístico no es fiable, pero la variable `incidencia_masiva` aparece ya recogida como lag en las tablas de soporte y facturación, así que su efecto está capturado de forma indirecta en el modelo.


### H5 — ¿La calidad de red de la zona está relacionada con el churn del cliente?

Esta es la hipótesis más importante del notebook porque conecta la calidad de red directamente con el abandono. Para analizarla, cruzamos la calidad media por zona con el perfil del cliente y su etiqueta de churn.


In [ ]:
# Calidad media por zona (sobre todo el período)
calidad_zona = calidad_clean.groupby('zona_id').agg(
    calidad_global_media  = ('indice_calidad_global',  'mean'),
    calidad_global_min    = ('indice_calidad_global',  'min'),
    cobertura_4g_media    = ('cobertura_4g_pct',       'mean'),
    cobertura_5g_media    = ('cobertura_5g_pct',       'mean'),
    latencia_media        = ('latencia_ms',            'mean'),
    velocidad_media       = ('velocidad_media_mbps',   'mean'),
    tasa_cortes_media     = ('tasa_cortes_pct',        'mean'),
    n_incidencias_zona    = ('incidencia_masiva',      'sum'),
).reset_index()

# Target: ever_churn por cliente
churn_agg = churn.groupby('cliente_id')['churn'].max().reset_index()
churn_agg.columns = ['cliente_id', 'ever_churn']

# Tabla analítica: cliente + zona + calidad + churn
df = (clientes[['cliente_id', 'zona_id']]
      .merge(churn_agg,     on='cliente_id', how='inner')
      .merge(calidad_zona,  on='zona_id',    how='left'))

df['churn_label'] = df['ever_churn'].map({0: 'No Churn', 1: 'Churn'})

print(f'Tabla analítica: {df.shape[0]:,} clientes')
print(f'Ever churn: {df["ever_churn"].mean()*100:.1f}%')
display(df.head())

In [ ]:
vars_calidad_cliente = ['calidad_global_media', 'cobertura_5g_media',
                        'latencia_media', 'tasa_cortes_media']

fig, axes = plt.subplots(1, 4, figsize=(18, 5))

for i, var in enumerate(vars_calidad_cliente):
    data_plot = df.dropna(subset=[var]).copy()
    sns.boxplot(data=data_plot, x='churn_label', y=var,
                order=['No Churn', 'Churn'],
                palette=PAL, ax=axes[i])
    axes[i].set_title(f'{var}\nvs Churn', fontweight='bold')
    axes[i].set_xlabel('')

plt.suptitle('H5 — Calidad de red de la zona vs Churn del cliente',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Tests estadísticos
print('Tests Mann-Whitney:')
for var in vars_calidad_cliente:
    test_mannwhitney(df, var)

### Resultado H5

Este es el resultado más sorprendente del notebook: **los boxplots de H5 muestran muy poca diferencia** entre churners y no churners en las métricas de calidad de red.

La calidad global media es prácticamente idéntica (~40 en ambos grupos), la cobertura 5G tampoco difiere, y aunque la latencia y la tasa de cortes son ligeramente mayores en churners, la diferencia es pequeña.

¿Significa esto que la calidad de red no importa? No exactamente. Hay varias explicaciones posibles:

1. **El efecto es indirecto**: la calidad de red no causa churn directamente, sino a través de más llamadas a soporte, más impagos y más insatisfacción. Eso ya lo vimos en los EDA anteriores con el `stress_calidad_lag`.
2. **La agregación pierde información**: estamos usando la calidad media de la zona en todo el período. Un cliente puede haber vivido un momento puntual muy malo que le llevó a irse, pero eso se diluye en la media.
3. **El cruce por zona es una aproximación**: dos clientes en la misma zona pueden tener experiencias muy distintas.

La conclusión es que **la calidad de red tiene más valor como variable temporal** (con lags) que como variable estática agregada.


---
## 5. Clasificación simple — Validación cuantitativa

Hasta aquí hemos analizado las hipótesis de forma visual y con tests estadísticos. Ahora vamos un paso más allá: entrenamos un modelo simple usando **solo las variables de calidad de red** para ver si son capaces de predecir el churn.

Usamos dos modelos:
- **Regresión logística**: modelo lineal, fácil de interpretar
- **Árbol de decisión**: captura relaciones no lineales

La métrica que usamos es el **AUC-ROC**: mide qué tan bien separa el modelo los churners de los no churners. Un AUC de 0.5 es igual que lanzar una moneda, 1.0 es perfecto.

⚠️ **Importante**: este modelo es solo para validar si las variables tienen poder predictivo. No es el modelo final del proyecto.


In [ ]:
vars_modelo = ['calidad_global_media', 'calidad_global_min', 'cobertura_4g_media',
               'cobertura_5g_media', 'latencia_media', 'velocidad_media',
               'tasa_cortes_media', 'n_incidencias_zona']

# Preparamos los datos
df_modelo = df.dropna(subset=vars_modelo + ['ever_churn'])
X = df_modelo[vars_modelo]
y = df_modelo['ever_churn']

print(f'Clientes para el modelo: {len(X):,}')
print(f'Tasa de churn: {y.mean()*100:.1f}%\n')

# Regresión logística
pipe_lr = Pipeline([
    ('scaler', StandardScaler()),
    ('model',  LogisticRegression(random_state=42, max_iter=1000))
])
auc_lr = cross_val_score(pipe_lr, X, y, cv=5, scoring='roc_auc').mean()

# Árbol de decisión
pipe_dt = Pipeline([
    ('model', DecisionTreeClassifier(max_depth=4, random_state=42))
])
auc_dt = cross_val_score(pipe_dt, X, y, cv=5, scoring='roc_auc').mean()

print(f'AUC Regresión Logística: {auc_lr:.3f}')
print(f'AUC Árbol de Decisión:   {auc_dt:.3f}')

# Importancia de variables (árbol)
pipe_dt.fit(X, y)
importancias = pd.Series(
    pipe_dt.named_steps['model'].feature_importances_,
    index=vars_modelo
).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 5))
importancias.plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('Importancia de variables — Árbol de decisión\n(solo variables de calidad de red)',
             fontweight='bold')
ax.set_xlabel('Importancia relativa')
plt.tight_layout()
plt.show()

### Resultado clasificación

- **AUC Regresión Logística: 0.596**
- **AUC Árbol de Decisión: 0.594**

Un AUC de ~0.60 con solo variables de calidad de red es un resultado moderado pero consistente con lo que vimos en H5: estas variables tienen algo de poder predictivo pero no son suficientes por sí solas.

Lo más revelador es el gráfico de importancia de variables del árbol: **`calidad_global_media` acapara casi toda la importancia (>90%)**, mientras que el resto de variables apenas contribuyen. Esto sugiere que el índice sintético ya recoge bien la información de las variables individuales (cobertura, latencia, velocidad...), y que en el modelo final probablemente baste con incluir ese índice en lugar de todas las variables de red por separado.

En el modelo final, combinando estas variables con las de facturación, soporte y perfil del cliente, esperamos mejorar el AUC significativamente.


---
## 6. Conclusiones del EDA de Calidad de Red

**H1 — Confirmada**: Las zonas rurales tienen peor calidad en todos los indicadores. El índice de calidad global es más del doble en urbana_premium que en rural, y la cobertura 5G triplica. El tipo de zona es la variable geográfica más discriminante.

**H2 — Parcialmente confirmada**: Hay diferencias entre regiones, pero desaparecen al controlar por tipo de zona. Centro es la peor región, pero probablemente porque tiene más zonas rurales. Para el modelo, `tipo_zona` es más útil que `region`.

**H3 — Confirmada**: El 5G ha crecido del 10% al 43% entre 2023 y 2025, con mejora paralela del índice de calidad global y la velocidad. Esto refuerza el argumento de usar lags temporales en el modelo.

**H4 — Indicios positivos pero muestra insuficiente**: Las 5 incidencias masivas muestran el efecto esperado (más cortes, menos velocidad, peor calidad), pero no podemos sacar conclusiones estadísticas robustas. Su efecto está mejor capturado por el `stress_calidad_lag`.

**H5 — No confirmada como variable estática**: La calidad media de la zona no distingue bien churners de no churners. El efecto de la calidad de red sobre el churn es indirecto (a través de soporte e impagos) y temporal (mejor capturado con lags).

**Clasificación**: AUC ~0.60 con solo variables de red. El `indice_calidad_global` es la variable más importante con diferencia. En el modelo final se usará combinada con las otras fuentes.

### Variable recomendada para el modelo final
- `indice_calidad_global` de la zona (resume bien el resto)
- `stress_calidad_lag` de facturación/soporte (ya lo teníamos — captura el efecto temporal)
- `tipo_zona` del cliente (diferencia estructural rural vs urbano)

---
*Notebook elaborado como parte de la asignatura de Prácticas Aplicadas — Máster en Ciencia de Datos 2026*
